In [7]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[1] # go up n levels (adjust as needed)
sys.path.append(str(ROOT))

from config import PROJECT_ROOT, APT_ROOT
from apt_project import *

In [8]:
import json
import pandas as pd
from pprint import pprint
import matplotlib.pyplot as plt
import networkx as nx
import seaborn as sns
import numpy as np

In [9]:
# -------------------------------------------------------------
# 1. Count how many groups use each technique
# -------------------------------------------------------------
tech_group_counts = (
    group_techniques_df.groupby('technique_id')['group_id']
                       .nunique()
                       .reset_index(name='num_groups_using')
)

# -------------------------------------------------------------
# 2. Compute inverse frequency for each technique
# -------------------------------------------------------------
tech_group_counts['inv_freq'] = 1 / tech_group_counts['num_groups_using']

# -------------------------------------------------------------
# 3. Merge inverse frequency onto group_techniques_df
# -------------------------------------------------------------
group_tech_novelty = group_techniques_df.merge(
    tech_group_counts[['technique_id', 'inv_freq']],
    on='technique_id',
    how='left'
)

# -------------------------------------------------------------
# 4. Compute average inverse frequency per group
# -------------------------------------------------------------
novelty_df = (
    group_tech_novelty.groupby(['group_id', 'group_name'])['inv_freq']
                      .mean()
                      .reset_index(name='avg_inv_freq')
)

# -------------------------------------------------------------
# 5. Normalize the score to 0–1
# -------------------------------------------------------------
novelty_df['scaled_score'] = (
    (novelty_df['avg_inv_freq'] - novelty_df['avg_inv_freq'].min()) /
    (novelty_df['avg_inv_freq'].max() - novelty_df['avg_inv_freq'].min())
)

# -------------------------------------------------------------
# 6. Sort from most → least novel
# -------------------------------------------------------------
novelty_df = novelty_df.sort_values('scaled_score', ascending=False).reset_index(drop=True)

novelty_df


,group_id,group_name,avg_inv_freq,scaled_score
0,intrusion-set--96e239be-ad99-49eb-b127-3007b8c...,Equation,0.527778,1.000000
1,intrusion-set--277d2f87-2ae5-4730-a3aa-50c1fdf...,Strider,0.370370,0.694551
2,intrusion-set--a0cb9370-e39b-44d5-9f50-ef78e41...,Axiom,0.268560,0.496988
3,intrusion-set--d8bc9788-4f7d-41a9-9e9d-ee1ea18...,LAPSUS$,0.266856,0.493683
4,intrusion-set--461b8e25-8f4a-4ea2-a4a8-e39df7c...,UNC3886,0.253340,0.467455
...,...,...,...,...
163,intrusion-set--fed4f0a2-4347-4530-b0f5-6dfd49b...,Nomadic Octopus,0.025041,0.024441
164,intrusion-set--03506554-5f37-4f8f-9ce4-0e9f01a...,Elderwood,0.023149,0.020770
165,intrusion-set--fe98767f-9df8-42b9-83c9-004b1de...,PittyTiger,0.017693,0.010182
166,intrusion-set--62a64fd3-aaf7-4d09-a375-d6f8bb1...,TA459,0.016710,0.008275
